# Notebook 06 — Workflows

**Skills were verbs. Now we write a sentence.**

A workflow is a *reusable, versioned procedure* that composes skills under an
identity — business as code. This is the **ISOLATE** move at workflow scale:
each phase runs in its own scoped context and hands the *next* phase a small
artifact, not its whole transcript. That is the antidote to **distraction** and
**confusion** on a long task. The chain works because no single window has to
hold the whole job.

What we prove in this run:
1. A real multi-phase chain drops a **versioned artifact out of each phase**.
2. `plan → do → validate` is **the same shape** at tool / skill / workflow scale.
3. An **ad-hoc one-prompt** attempt vs the workflow on the same task — quality
   and repeatability diverge.
4. We **graduate** the ad-hoc run into a reusable workflow definition.

## Setup

One import wires the model, the `ask()` helper, and the data path. We import
only the chart helpers this notebook actually uses from `roadshow_viz` — every
visual below is one call into that module.

In [ ]:
from roadshow import setup
from roadshow_viz import compare, matrix, hbars
import json

model, ask, ntok, DATA = setup()

**Name the repo's real workflow assets.** "Business as code" lives on disk:
the 7 phase commands in `.claude/commands/`, identities in `.claude/agents/`,
shipped artifacts in `.claude/examples/`, and a scratch dir where each phase
drops its artifact.

In [ ]:
# The repo's real workflow assets — "business as code" on disk.
CLAUDE   = DATA.parent / ".claude"
COMMANDS = CLAUDE / "commands"            # the 7 phase commands
AGENTS   = CLAUDE / "agents"              # identities (principled-coder)
EXAMPLES = CLAUDE / "examples"            # real shipped artifacts
SCRATCH  = DATA / "scratch"
WF_DIR   = SCRATCH / "workflow"           # where each phase drops its artifact
WF_DIR.mkdir(parents=True, exist_ok=True)

print("artifact drop dir:", WF_DIR)

**Define `PHASES` — the lesson's source of truth.** Each entry is
`(phase, skill behind it, artifact it emits)`. Every cell below reads from this
list, so the seven-phase chain is defined exactly once.

In [ ]:
# The seven phases of the spec_dev workflow: each phase, the skill behind it,
# and the artifact it emits. Every cell below reads from this list.
PHASES = [
    ("steering",  "steering-docs-creator",      "product.md"),
    ("brainstorm","brainstorming",              "2026-06-02-since-flag.md"),
    ("build",     "build-decisions",            "decisions.md"),
    ("deepen",    "plan-deepener",              "research-insights.md"),
    ("specify",   "spec-prd-generator",         "PRD.md"),
    ("implement", "spec-execution",             "PATCH.diff"),
    ("review",    "architecture-adr-generator", "ADR-since-flag.md"),
]
PHASE_NAMES = [p[0] for p in PHASES]

print("phase commands available:", sorted(p.name for p in COMMANDS.glob("*.md")))

## ▶ Control cell — the only cell you edit

Every cell below reads these plain variables. Change a value here, then re-run
the cells beneath it — that is the whole interface.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  CONTROL CELL — edit, then re-run the cells below
# ════════════════════════════════════════════════════════════════════════
MODE             = "workflow"     # "ad_hoc" (one big prompt) | "workflow" (the phased chain)
WORKFLOW         = "spec_dev"     # which workflow to run — the 7-phase spec-development chain
STOP_AFTER_PHASE = "specify"      # stop the chain after this phase (keeps the demo bounded)
TASK             = "Add a --since flag to the audit-log exporter."   # the job the chain runs
OFFLINE          = False          # False -> run phases LIVE (real model + tool calls)
                                  # True  -> replay the committed traces (no network / air-gapped path)

# Label used to look up the precomputed traces and title the charts.
MODEL_LABEL = "anthropic"
print(f"MODE={MODE}  WORKFLOW={WORKFLOW}  STOP_AFTER_PHASE={STOP_AFTER_PHASE}  OFFLINE={OFFLINE}")

## The seven phases

| # | Phase | Skill / agent behind it | Artifact it emits |
|---|---|---|---|
| 1 | `steering`   | steering-docs-creator      | `product.md` (the bar) |
| 2 | `brainstorm` | brainstorming              | a brainstorm note (the WHY) |
| 3 | `build`      | build-decisions            | `decisions.md` (the choices) |
| 4 | `deepen`     | plan-deepener              | research insights (edge cases) |
| 5 | `specify`    | spec-prd-generator         | `PRD.md` (EARS requirements) |
| 6 | `implement`  | spec-execution             | a patch |
| 7 | `review`     | architecture-adr-generator | an ADR + 3Cs score |

The **`principled-coder`** identity holds the bar throughout — Simplicity →
Modularity → Security → Scalability filter every phase (callback to Section 5).
Each phase only receives the *small artifact* from the phase before it, never
the full transcript. That is isolation in practice.

### The run engine — the heart of the lesson

Two functions carry the whole notebook:

- **`run_phase(name, state) -> (artifact, new_state)`** runs **one** phase in its
  own scoped context. It builds the prompt from *only* the prior artifact + the
  phase command — never the full transcript. That is the **ISOLATE** move in code.
- **`solve(task, mode, stop_after)`** is the A/B driver: `ad_hoc` is one big
  prompt; `workflow` chains the phases, threading each small artifact forward.

When `OFFLINE=True`, `solve` replays a committed trace instead of calling the
model — same shapes, no network.

**Load the committed traces.** With `OFFLINE=True` we replay a real recorded
run with no network. `load_precomputed` returns one recorded run for the current
mode + model label.

In [ ]:
# Committed traces let OFFLINE=True replay a real run with no network.
TRACES = json.loads((SCRATCH / "precomputed_traces.json").read_text())

def _trace_key(mode):
    return f"{mode}__{MODEL_LABEL}"

def load_precomputed(mode, run_index=0):
    """Return one recorded run for (mode, current MODEL_LABEL)."""
    runs = TRACES[_trace_key(mode)]
    return runs[run_index % len(runs)]

print("traces loaded for:", sorted(TRACES.keys()))

**`run_phase` — the ISOLATE move in code.** Each phase sees only the phase
command + the **one** prior artifact + the task — never the whole transcript.
The live path runs a real `arcrun` loop whose single tool (`emit_artifact`)
captures the phase's artifact; the offline path reads the pre-dropped file.

In [ ]:
async def run_phase(name, state):
    """Run a single workflow phase in its own scoped context.

    LIVE PATH (OFFLINE=False): build the phase prompt from ONLY the prior
    artifact (state['last_artifact']) + the phase command, run a real arcrun
    agentic loop whose one tool (`emit_artifact`) writes the phase artifact to
    WF_DIR, and thread the live cost/turns forward. Returns (path, new_state).
    OFFLINE PATH: read the pre-dropped artifact for this phase.

    This is the ISOLATE move: each phase sees only the command + the prior
    artifact + the task — never the whole transcript.
    """
    skill = dict((p[0], p[1]) for p in PHASES)[name]
    fname = dict((p[0], p[2]) for p in PHASES)[name]
    artifact_path = WF_DIR / f"{name}__{fname}"

    if not OFFLINE:
        # arcrun is the real agentic loop. The 2nd positional arg is a CAPABILITY
        # PROVIDER (StaticProvider([...])), not tools=. The loop requires >=1
        # capability, so each phase gets one tool that captures its artifact —
        # that also gives us a real tool_calls_made count to show the room.
        from arcrun import run as arc_run, StaticProvider, Tool, ToolContext

        captured = {}
        async def emit_artifact(args: dict, ctx: ToolContext) -> str:
            captured["content"] = args.get("content", "")
            return f"{fname} saved."
        emit_tool = Tool(
            name="emit_artifact",
            description=f"Save the finished {fname} artifact. Call exactly once with the full artifact body.",
            input_schema={"type": "object",
                          "properties": {"content": {"type": "string", "description": "the artifact body"}},
                          "required": ["content"]},
            execute=emit_artifact,
        )

        # Map phase name -> its real command filename on disk. Most phases share
        # their name (build.md, specify.md), but `steering` ships as
        # create-steering-docs.md, so look it up rather than guessing steering.md.
        cmd_filename = {"steering": "create-steering-docs"}.get(name, name)
        cmd = (COMMANDS / f"{cmd_filename}.md")
        cmd_text = cmd.read_text() if cmd.exists() else f"# /{name}"
        prior = state.get("last_artifact_text", "(none — this is the first phase)")
        prompt = (f"You are the {skill} skill executing the /{name} phase.\n"
                  f"Command:\n{cmd_text[:1500]}\n\nPrior artifact:\n{prior[:2000]}\n\n"
                  f"Task: {state['task']}\n"
                  f"Write the {fname} artifact, then call emit_artifact with its full content.")
        # TOP-LEVEL await chain: run_phase is async, awaited at top level by the caller.
        result = await arc_run(model, StaticProvider([emit_tool]),
                               "Identity: principled-coder.", prompt, max_turns=6)
        artifact_path.write_text(captured.get("content") or result.content)
        # Thread the live telemetry forward so the driver reports REAL numbers.
        state.setdefault("_live", {"turns": 0, "tool_calls": 0, "cost_usd": 0.0})
        state["_live"]["turns"]      += result.turns
        state["_live"]["tool_calls"] += result.tool_calls_made
        state["_live"]["cost_usd"]   += (result.cost_usd or 0.0)

    text = artifact_path.read_text() if artifact_path.exists() else "(artifact missing)"
    new_state = dict(state)
    new_state["last_artifact_text"] = text
    new_state.setdefault("artifacts", []).append({"phase": name, "skill": skill, "file": artifact_path.name})
    return artifact_path, new_state

**`solve` — the A/B driver.** `ad_hoc` replays one big "just do it" prompt
(no phases, no gates). `workflow` runs the phases up to `stop_after`, threading
each small artifact forward and reporting the **real** accumulated telemetry
when it ran live.

In [ ]:
async def solve(task, mode="workflow", stop_after="specify"):
    """A/B driver. ad_hoc = one prompt; workflow = the phased chain to stop_after.

    The `workflow` path runs LIVE arcrun loops when OFFLINE=False (real cost,
    real tool calls) and falls back to the committed trace when OFFLINE=True.
    The rubric/fractal scales always come from the recorded trace (they are the
    model-dependent statistical artifacts we replay for the air-gapped room).
    """
    if mode == "ad_hoc":
        # One big "just do it" prompt — no phases, no gates, no artifacts.
        tr = load_precomputed("ad_hoc")
        return {"mode": "ad_hoc", "content": tr["content"], "artifacts": [],
                "phases_run": tr["phases_run"], "rubric": tr["rubric"],
                "turns": tr["turns"], "cost_usd": tr["cost_usd"], "fractal": tr.get("fractal")}

    # workflow: run phases up to stop_after, threading the small artifact forward.
    state = {"task": task, "artifacts": []}
    idx = PHASE_NAMES.index(stop_after)
    for name in PHASE_NAMES[: idx + 1]:
        _, state = await run_phase(name, state)
    tr = load_precomputed("workflow")  # rubric/fractal come from the recorded run
    live = state.get("_live")
    # When we ran live, report the REAL accumulated telemetry; else replay the trace.
    turns = live["turns"] if live else tr["turns"]
    cost  = live["cost_usd"] if live else tr["cost_usd"]
    return {"mode": "workflow", "content": tr["content"], "artifacts": state["artifacts"],
            "phases_run": PHASE_NAMES[: idx + 1], "rubric": tr["rubric"],
            "turns": turns, "cost_usd": cost,
            "tool_calls_made": (live["tool_calls"] if live else None),
            "live": bool(live), "fractal": tr.get("fractal")}

print("engine ready: run_phase(), solve()  (async — await them at top level)")

## Ad-hoc vs workflow — run both (A/B contrast)

Same task, two harnesses. Ad-hoc produces *something* but skips the spec, the
edge cases, the review. The workflow produces a steering doc, a brainstorm, a
spec — **reviewable artifacts**. One loop, not duplicated cells.

In [ ]:
RESULTS = {}
# solve() is async (the workflow path awaits a real arcrun loop per phase).
# Top-level await works in the notebook kernel — no asyncio.run() needed.
for mode in ["ad_hoc", "workflow"]:
    RESULTS[mode] = await solve(TASK, mode=mode, stop_after=STOP_AFTER_PHASE)

**Read the two runs side by side.** Phases, artifacts emitted, a snippet of
the output, and the real telemetry (turns / cost / tool calls).

In [ ]:
for mode, r in RESULTS.items():
    tag = "LIVE arcrun" if r.get("live") else "replayed trace"
    print(f"=== {mode} ({tag}) ===")
    print("  phases:", r["phases_run"])
    print("  artifacts emitted:", len(r["artifacts"]))
    print("  output:", r["content"][:160].replace("\n", " "))
    line = f"  turns={r['turns']}  cost=${r['cost_usd']:.4f}"
    if r.get("tool_calls_made") is not None:
        line += f"  tool_calls_made={r['tool_calls_made']}"
    print(line)
    print()

## The artifacts appear

Tangible proof: *this phase produced this file.* Each is small, versioned, and
reviewable. The ad-hoc run, by contrast, is a single undifferentiated blob.

In [ ]:
wf = RESULTS["workflow"]
print(f"Workflow run dropped {len(wf['artifacts'])} artifacts under {WF_DIR.relative_to(DATA.parent)}:\n")
for a in wf["artifacts"]:
    p = WF_DIR / a["file"]
    body = p.read_text()
    head = "\n".join(body.splitlines()[:4])
    print(f"--- {a['phase']:<10s} [{a['skill']}] -> {a['file']}")
    print("   " + head.replace("\n", "\n   "))
    print()

print("=== ad_hoc ===  (no checkpointed artifacts — one blob)")
print(RESULTS["ad_hoc"]["content"])

## Visual 1 — structure: many small artifacts vs one blob

The point isn't pixels on a timeline — it's the **structural contrast**. The
workflow drops *N checkpointed artifacts*; the ad-hoc run produces *one*
undifferentiated blob. One `compare()` call makes the gap read instantly.

In [ ]:
structure = {
    "ad-hoc (1 blob)":        len(RESULTS["ad_hoc"]["artifacts"]) or 1,
    "workflow (N artifacts)": len(RESULTS["workflow"]["artifacts"]),
}
compare(structure,
        title="Visual 1 — A versioned artifact drops out of each workflow phase",
        ylabel="checkpointed artifacts")

## Isolation — why the chain doesn't choke on a long job

This is the thesis of the whole section: **no single window has to hold the
whole job.** Each phase's context is *only* the phase command + the **one** prior
artifact (capped) + the task — never the accumulated transcript.

We compute the **peak context window** the isolated chain ever holds, then
compare it to the **monolith** — cramming every artifact into one window. The
isolated peak should be a fraction of the monolith's. That gap is the
**distraction / confusion** antidote, measured.

In [ ]:
# Reconstruct each phase's REAL context window using the SAME budget run_phase
# uses live: command (<=1500 chars) + the single prior artifact (<=2000 chars).
# The isolated peak is the largest single window the chain ever holds.
cmd_file = {"steering": "create-steering-docs", "brainstorm": "brainstorm",
            "build": "build", "deepen": "deepen", "specify": "specify"}

iso_windows, mono_total, prior_text = [], 0, ""
for a in RESULTS["workflow"]["artifacts"]:
    name = a["phase"]
    cmd = COMMANDS / f"{cmd_file.get(name, name)}.md"
    cmd_text = cmd.read_text()[:1500] if cmd.exists() else ""
    window = cmd_text + prior_text[:2000]          # exactly what the phase sees
    iso_windows.append((name, ntok(window)))
    art_text = (WF_DIR / a["file"]).read_text()
    mono_total += ntok(art_text)                   # monolith accumulates EVERY artifact
    prior_text = art_text                          # next phase sees only THIS artifact

iso_peak = max(t for _, t in iso_windows)
print("per-phase isolated context window (command + ONE prior artifact):")
for name, t in iso_windows:
    print(f"   {name:<10s} {t:>6d} tok")
print(f"\nISOLATED peak window : {iso_peak:>6d} tok  (largest any single phase holds)")
print(f"MONOLITH peak window : {mono_total:>6d} tok  (all {len(iso_windows)} artifacts in ONE window)")
print(f"\n-> isolation keeps the peak window {mono_total / iso_peak:.0f}x smaller. "
      f"That is the distraction/confusion antidote, in tokens.")

**Chart it.** The isolated peak window vs the monolith — one `compare()`
call, tokens on the y-axis.

In [ ]:
compare({"isolated\n(peak phase window)": iso_peak,
         "monolith\n(all artifacts at once)": mono_total},
        title="Isolation — the peak window never has to hold the whole job",
        ylabel="peak context window (tokens)", fmt="{:.0f} tok")

## `plan / do / validate` is fractal

Watch the same three-beat pattern at three scales **inside this one run**. The
pattern isn't ours — it's the shape of the published agent loops:
- **reason → act → observe** is [ReAct (Yao et al., 2022)](https://arxiv.org/abs/2210.03629)
- the **act → evaluate → revise** beat is [Reflexion (Shinn et al., 2023)](https://arxiv.org/abs/2303.11366)

We're claiming the *self-similarity across scales*, not inventing the loop.

> **Honest caveat:** "fractal" is a pedagogical analogy, not a formal claim — the
> three scales *rhyme*, they aren't literally identical.

### Instrument the three scales

Read from the recorded run's `fractal` trace: one tool call's
reason→act→observe, one skill's load→steps→validate, and the workflow's
plan→execute→review (review = the Reflexion beat). Printed aligned so the rhyme
is obvious.

In [ ]:
fr = RESULTS["workflow"]["fractal"]
if fr is None:  # ad-hoc has no phased trace; pull the workflow trace for the scales
    fr = load_precomputed("workflow")["fractal"]

beats = ["plan", "do", "validate"]
scales = ["tool", "skill", "workflow"]
colw = 40
print(f"(scales harvested from a real {fr['_model']} run)\n")
header = " " * 10 + "".join(b.upper().ljust(colw) for b in beats)
print(header)
print("-" * len(header))
for s in scales:
    row = fr[s]
    line = f"{s:<10s}" + "".join(str(row[b])[:colw-1].ljust(colw) for b in beats)
    print(line)
print("\nSame three beats. Three scales. They rhyme.")

## Visual 2 — the fractal (hero visual)

Three scales × three beats. Every cell is present — the **same**
`plan → do → validate` shape exists at the tool, skill, and workflow scale.
One `matrix()` call shows the self-similarity as a full grid.

> **These numbers are illustrative, not measured live.** The grid below is hardcoded to all-present (`[[1,1,1],[1,1,1],[1,1,1]]`) to assert that the `plan -> do -> validate` shape *exists* at each scale — it does not chart a quantity measured from this run.

In [ ]:
matrix(row_labels=["tool", "skill", "workflow"],
       col_labels=["plan", "do", "validate"],
       grid=[[1, 1, 1], [1, 1, 1], [1, 1, 1]],
       title="Visual 2 — plan -> do -> validate: the same shape at every scale")

## Checkpoints — where humans plan and validate

Between phases is where you stay in control. Let's gate one: the spec must pass
the `spec-validation` **3Cs** rubric before `implement` is allowed to start.

**The AUTO checkpoint — score the spec on the 3Cs rubric.** A transparent
keyword rubric (no model needed) scores Completeness / Consistency /
Correctness, normalizes each to 0-10, and unlocks `implement` only on a pass.

In [ ]:
# AUTO checkpoint: the spec must pass the 3Cs rubric (Completeness / Consistency /
# Correctness) before `implement` runs. Transparent keyword-rubric fallback so it
# scores with no model — the same gate the spec-validation skill enforces.
def score_3cs(spec_text):
    t = spec_text.lower()
    completeness = sum(k in t for k in ["req-", "nfr", "must", "should"])          # has requirements
    consistency  = sum(k in t for k in ["when", "shall"])                          # EARS form, no contradiction markers
    correctness  = sum(k in t for k in ["utc", "stream", "backwards", "exit"])     # edge cases addressed
    # normalize each to 0-10
    c1 = min(10, completeness * 3); c2 = min(10, consistency * 5); c3 = min(10, correctness * 3)
    return {"Completeness": c1, "Consistency": c2, "Correctness": c3, "overall": round((c1 + c2 + c3) / 3, 1)}

spec_path = WF_DIR / "specify__PRD.md"
scores = score_3cs(spec_path.read_text())
PASS_THRESHOLD = 7.0
passed = scores["overall"] >= PASS_THRESHOLD
print("3Cs gate on the spec:", scores)
print(f"  -> {'PASS' if passed else 'FAIL'} (threshold {PASS_THRESHOLD}) "
      f"=> implement is {'UNLOCKED' if passed else 'BLOCKED'}")

**The HUMAN checkpoint.** `STOP_AFTER_PHASE` halts the chain for approval —
no widget, it just stops and tells the presenter how to continue.

In [ ]:
# HUMAN checkpoint: STOP_AFTER_PHASE halts the chain for approval. No widget — it
# just stops and tells the presenter how to continue.
idx = PHASE_NAMES.index(STOP_AFTER_PHASE)
remaining = PHASE_NAMES[idx + 1:]
if remaining:
    print(f"Human checkpoint: chain stopped after '{STOP_AFTER_PHASE}'. "
          f"Awaiting approval before: {remaining}.")
    print(f"  -> set STOP_AFTER_PHASE = '{remaining[-1]}' (or later) and re-run to continue.")
else:
    print(f"Human checkpoint: '{STOP_AFTER_PHASE}' is the last phase — chain complete.")

## Visual 3 — quality: ad-hoc vs workflow

Score both runs against a rubric (spec completeness, edge cases, review
performed, artifact count), each mode **run twice** to show the spread. The
honest expectation: the *workflow* scores higher **and** drifts less because the
gates clamp it.

> **State the model-dependence out loud:** the *size* of the gap is not a fixed
> number — it depends on task and model, and on a trivial task the gap can
> collapse. The claim is **directional** (structure adds repeatability), not
> "workflow is always N× better." This is the day's thesis made local: a *good
> harness + a decent model beats a weak harness + a frontier model* — same
> weights, different scaffold (one vendor reports
> [scaffolding swings of 20+ pts on SWE-bench](https://particula.tech/blog/agent-scaffolding-beats-model-upgrades-swe-bench),
> a single-source illustration rather than an established benchmark figure).
> If `MODEL="ollama"`, this is the most valuable cell in the room: the local 8B
> model **with the workflow** should beat itself ad-hoc.

> **These numbers are replayed, not measured live this run.** The rubric scores come from the committed `precomputed_traces.json`, not from grading this session's output. Treat the bars as an illustrative, recorded contrast.

**Compute the rubric means and spread.** Average each mode's two repeats
across the four rubric dimensions; the spread (max − min) is the variance the
gates are supposed to clamp.

In [ ]:
dims = ["spec_completeness", "edge_cases_covered", "review_performed", "artifact_count"]

def mean_and_spread(mode):
    runs = TRACES[f"{mode}__{MODEL_LABEL}"]
    totals = [sum(r["rubric"][d] for d in dims) for r in runs]   # total rubric score per repeat
    mean = sum(totals) / len(totals)
    spread = max(totals) - min(totals)
    return mean, spread

for mode in ["ad_hoc", "workflow"]:
    m, s = mean_and_spread(mode)
    print(f"{mode:<10s} mean total rubric = {m:.1f}   spread over 2 repeats = {s}")

print("\nDirectional, not absolute: the workflow's gates clamp variance; ad-hoc drifts.")
print("On a trivial task the gap can shrink — the structure is what makes it repeatable.")

**Chart the contrast.** Mean total rubric score per mode — one `compare()`
call. (The per-repeat spread is printed above; the chart shows the directional
quality gap.)

In [ ]:
quality = {
    "ad-hoc":   mean_and_spread("ad_hoc")[0],
    "workflow": mean_and_spread("workflow")[0],
}
compare(quality,
        title=f"Visual 3 — Quality ({MODEL_LABEL}): mean total rubric score",
        ylabel="rubric score (sum of 4 dims)", fmt="{:.1f}")

## Workflows are portable artifacts

This whole process is a folder you can copy.

In [ ]:
# The workflow is plain text under version control: commands + agents + skills +
# examples. `cp -R .claude/ your-project/` gives any repo the same process.
def tree(root, prefix="", max_entries=8):
    entries = sorted([p for p in root.iterdir() if not p.name.startswith(".")])[:max_entries]
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(f"{prefix}{'└── ' if last else '├── '}{p.name}{'/' if p.is_dir() else ''}")
        if p.is_dir() and prefix.count("    ") < 1:
            kids = sorted(p.iterdir())[:6]
            for j, k in enumerate(kids):
                kl = j == len(kids) - 1
                print(f"{prefix}{'    ' if last else '│   '}{'└── ' if kl else '├── '}{k.name}")

print(".claude/")
tree(CLAUDE)
print()
print("Lab angle:")
print("  - Plain text + version control => auditable & reproducible: every phase")
print("    artifact is checked in, so you can answer 'which version of which")
print("    context produced this decision?'")
print("  - Carries across an air-gap with no service dependency: the same .claude/")
print("    folder drives the chain whether MODEL is a hosted API or a local Ollama")
print("    model on a classified network.")
print("  - Business process as a versioned, portable artifact:  cp -R .claude/ your-project/")

## Graduate an ad-hoc run into a reusable workflow (autobrowse preview)

You just ran something ad-hoc that worked. Let's **harvest** it into a reusable
workflow — the manual version of what Section 7 automates.

This is a real, current research direction, not a metaphor: synthesizing
reusable workflows from past agent traces is exactly what [ReUseIt (arXiv
2510.14308, 2025)](https://arxiv.org/abs/2510.14308) does (it also adds
*condition checks* and *fallback actions* harvested from failed attempts), and
the "run it, study the trace, graduate the winner into a reusable skill" loop is
what Browserbase's [Autobrowse](https://www.browserbase.com/blog/autobrowse)
describes. Ours is a deliberately simple stub of that idea.

The ad-hoc run produced *one blob* — there is no sequence to harvest from it.
The **workflow** run is the one that worked *and* left a trace: an ordered list
of `(phase, skill, artifact)` steps. That structured winner is what graduates
into a reusable definition.

**Define the per-phase condition checks.** Each harvested phase carries a
`check` — the condition that must hold for the phase to count as done (the
ReUseIt move).

In [ ]:
# Per-phase condition CHECK — what must hold for a phase to count as done.
CHECKS = {"steering":  "product.md names the bar (pillars + scope)",
          "brainstorm":"at least one WHY / use-case captured",
          "build":     "every decision has a rationale",
          "deepen":    "edge cases enumerated",
          "specify":   "spec has >=1 Must requirement",
          "implement": "patch applies cleanly; tests added",
          "review":    "3Cs >= pass threshold"}

**`harvest_workflow` — derive a reusable stub from a recorded run's trace.**
Read the ordered `(phase, skill, artifact)` steps the run actually took, attach
a check per phase, and a fallback on the terminal gate. The stub is *derived*
from the trace, not pasted in.

In [ ]:
def harvest_workflow(trace):
    """Derive a reusable workflow stub from the steps a successful run recorded."""
    artifacts = trace.get("artifacts", [])
    stub = {"name": "audit_export_review",
            "harvested_from": f"{trace['mode']} trace ({len(artifacts)} recorded steps)",
            "phases": []}
    for a in artifacts:                       # the real ordered sequence the run took
        entry = {"phase": a["phase"], "skill": a["skill"],
                 "artifact": a["file"].split("__", 1)[-1],
                 "check": CHECKS.get(a["phase"], "phase produced an artifact")}
        stub["phases"].append(entry)
    # Enrich the LAST step with a fallback harvested from failed attempts (ReUseIt).
    if stub["phases"]:
        stub["phases"][-1]["fallback"] = "if check FAILS -> loop back to specify"
    return stub

**Harvest from the workflow run** — the successful trace that left a real
sequence — and print the derived reusable stub.

In [ ]:
# Harvest from the WORKFLOW run — the successful trace that left a real sequence.
stub = harvest_workflow(RESULTS["workflow"])

print("Harvested workflow stub (DERIVED from the recorded trace):\n")
print(json.dumps(stub, indent=2))
print(f"\n{len(stub['phases'])} phases harvested directly from the run's recorded steps "
      f"-> a reusable procedure. Foreshadows Section 7, which automates this harvest.")

### Visual 4 — trace → workflow harvest

The harvest pipeline as an ordered sequence: a recorded run's steps become a
reusable phase chain with checks and a fallback. One `hbars()` call lists the
five steps top-to-bottom, with the final "reusable stub" step highlighted.

In [ ]:
harvest_steps = {
    "1. recorded run trace (it worked)":       1,
    "2. read ordered (phase, skill) steps":    1,
    f"3. name the {len(stub['phases'])} phases + skills": 1,
    "4. add checks + fallback (from a failure)": 1,
    "5. reusable workflow stub":               1,
}
hbars(harvest_steps,
      title="Visual 4 — Harvest: a successful trace graduates into a reusable workflow",
      xlabel="harvest pipeline (ReUseIt / Autobrowse)",
      highlight="5. reusable workflow stub")

## ✅ Your turn — compose your own workflow

Pick a repeatable lab job, list the phases, map each to a skill (real or stub),
and run it up to the first checkpoint. This forces the
compose-skills-into-a-procedure muscle.

In [ ]:
# ✅ TODO — compose your own workflow.

# 1. Name a repeatable lab job (one sentence).
MY_JOB = "TODO — e.g. ingest a COA, validate it, update ERP, alert on out-of-spec"

# 2. List the phases (ordered) and the skill each phase needs (real or stub name).
MY_WORKFLOW = [
    # (phase_name, skill_name, artifact_it_emits)
    ("ingest",   "TODO-skill", "raw_coa.json"),
    ("validate", "TODO-skill", "validation_report.md"),
    ("update",   "TODO-skill", "erp_update.diff"),
    ("alert",    "TODO-skill", "alert.md"),
]

# 3. Where does the FIRST human/auto checkpoint go? (which phase must pass before the next runs)
MY_FIRST_CHECKPOINT = "validate"   # TODO — e.g. spec passes 3Cs, COA passes spec-limits

# 4. Run it up to the first checkpoint (offline stub: just lists what each phase emits).
print(f"Job: {MY_JOB}\n")
for phase, skill, artifact in MY_WORKFLOW:
    print(f"  phase={phase:<10s} skill={skill:<14s} -> emits {artifact}")
    if phase == MY_FIRST_CHECKPOINT:
        print(f"  -- CHECKPOINT: '{phase}' must pass before the rest run. Stopping here for approval. --")
        break

# When you have real skills, swap the stub names above and set OFFLINE=False to run live.

## Takeaway

> A **workflow** composes skills under an identity into a **repeatable,
> versioned, auditable procedure** — business as code — and `plan / do / validate`
> is the **same shape all the way down**. You've now built the whole stack by
> hand. Next: how the stack *improves itself* — **adaptation**.